# Classical Tobit on the Fair Affairs Dataset

The Fair (1978) extramarital-affairs dataset is a canonical Tobit example: the dependent variable `affairs` is the number of affairs per year, left-censored at zero (about 68% of respondents report zero affairs). We use the censored regression with `left=0` and no right threshold — the classical Type-1 Tobit.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from censtrunc import CensoredRegression

data = sm.datasets.fair.load_pandas().data
print('n =', len(data))
print(f"share with affairs == 0: {(data['affairs'] == 0).mean():.1%}")
data.head()

n = 6366
share with affairs == 0: 67.8%


,rate_marriage,age,yrs_married,children,religious,educ,occupation,occupation_husb,affairs
0,3.0,32.0,9.0,3.0,3.0,17.0,2.0,5.0,0.111111
1,3.0,27.0,13.0,3.0,1.0,14.0,3.0,4.0,3.230769
2,4.0,22.0,2.5,0.0,1.0,16.0,3.0,5.0,1.400000
3,4.0,37.0,16.5,4.0,3.0,16.0,5.0,5.0,0.727273
4,5.0,27.0,9.0,1.0,1.0,14.0,3.0,4.0,4.666666


## Specification

Following Fair (1978), we regress the count of affairs on the standard set of demographic and marital variables.

In [2]:
covariates = ['rate_marriage', 'age', 'yrs_married', 'children',
              'religious', 'educ', 'occupation', 'occupation_husb']
X = data[covariates]
y = data['affairs']

## Fit

We tell `CensoredRegression` that the left threshold is zero; the right threshold is left unspecified.

In [3]:
model = CensoredRegression(left=0.0).fit(X, y)
print(model.summary())

                              Censored Regression Results                               
Dep. Variable:           y                   No. Observations:        6366
Model:                   CensoredRegression  Df Model:                9
Method:                  MLE                 Df Residuals:            6356
Date:                    Fri, 29 May 2026    Log-Likelihood:          -7804.3803
Time:                    22:38:23            LL-Null:                 -8155.0240
AIC:                     15628.7605          LLR p-value:             3.780e-146
BIC:                     15696.3478          Pseudo R-squ.:           0.0430
Left threshold:          0                   Right threshold:         none
                      coef     std err         z     P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
sigma               4.4990      0.0771   58.3463    0.0000      4.3478      4.6501
const               7.8354      0.7135   10.9

## Marginal effects

Because the response is left-censored at zero, the censored marginal effects are smaller in absolute value than the latent ones. The latent (uncensored) coefficients describe the propensity scale, while the censored marginal effects describe the *expected number* of affairs.

In [4]:
ame_cens = model.ame(kind='censored').to_dataframe()
ame_lat  = model.ame(kind='latent').to_dataframe()
pd.concat([
    ame_lat[['dy/dx']].rename(columns={'dy/dx': 'latent'}),
    ame_cens[['dy/dx', 'std err', 'P>|z|']].rename(columns={'dy/dx': 'censored E[Y]'}),
], axis=1).round(4)

,latent,censored E[Y],std err,P>|z|
rate_marriage,-1.5313,-0.4326,0.0213,0.0000
age,-0.1051,-0.0297,0.0070,0.0000
yrs_married,0.1282,0.0362,0.0074,0.0000
children,-0.0276,-0.0078,0.0218,0.7206
religious,-0.9434,-0.2665,0.0243,0.0000
educ,-0.0857,-0.0242,0.0106,0.0228
occupation,0.3125,0.0883,0.0232,0.0001
occupation_husb,0.0143,0.0040,0.0157,0.7966


## Sanity check vs. OLS

Naive OLS on the censored sample understates the slope coefficients (Greene 1981). The Tobit MLE corrects this.

In [5]:
X_int = sm.add_constant(X)
ols = sm.OLS(y, X_int).fit()
pd.DataFrame({
    'OLS coef':    ols.params.values,
    'Tobit coef':  model.coef_,
}, index=ols.params.index).round(4)

,OLS coef,Tobit coef
const,3.6235,7.8354
rate_marriage,-0.4205,-1.5313
age,-0.0146,-0.1051
yrs_married,-0.0160,0.1282
children,-0.0171,-0.0276
religious,-0.2437,-0.9434
educ,-0.0174,-0.0857
occupation,0.0658,0.3125
occupation_husb,0.0040,0.0143
